# Day 44: Deploy vLLM, Benchmark TTFT and Throughput

**Layer:** Implementation


## Concept Overview

vLLM is the standard production LLM serving framework. Deploying it correctly requires tuning tensor parallelism, memory utilization, and quantization for the target model. Benchmarking with realistic traffic patterns reveals the throughput-latency tradeoff.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. vLLM Deployment and Configuration

vLLM serves models via an OpenAI-compatible REST API. Key configuration: tensor_parallel_size, gpu_memory_utilization, max_model_len, quantization.


In [ ]:
print('vLLM deployment configuration guide:')
print()
configs = [
    ('Small model (1-7B)',  {'tp': 1, 'gpu_mem': 0.90, 'max_len': 32768, 'quant': 'none'}),
    ('Medium model (8-13B)',{'tp': 1, 'gpu_mem': 0.90, 'max_len': 8192,  'quant': 'awq'}),
    ('Large model (70B)',   {'tp': 2, 'gpu_mem': 0.85, 'max_len': 4096,  'quant': 'awq'}),
    ('MoE (Mixtral)',       {'tp': 2, 'gpu_mem': 0.85, 'max_len': 32768, 'quant': 'none'}),
]
for name, cfg in configs:
    print(f'{name}:')
    print(f'  python -m vllm.entrypoints.openai.api_server \\')
    print(f'    --model <model_path> \\')
    print(f'    --tensor-parallel-size {cfg["tp"]} \\')
    print(f'    --gpu-memory-utilization {cfg["gpu_mem"]} \\')
    print(f'    --max-model-len {cfg["max_len"]} \\')
    print(f'    --quantization {cfg["quant"]}')
    print()


## 2. Benchmarking TTFT and Throughput

Standard benchmarks: vllm/benchmarks/benchmark_serving.py measures TTFT, TPOT, and request throughput under load.


In [ ]:
import asyncio, time, numpy as np

async def simulate_vllm_benchmark(n_requests=100, concurrency=10, prompt_len=256, output_len=128):
    """Simulate benchmark_serving.py behavior."""
    semaphore = asyncio.Semaphore(concurrency)
    ttfts, tpots = [], []

    async def single_request():
        async with semaphore:
            # Simulate TTFT (prefill) + decode
            ttft = np.random.lognormal(np.log(prompt_len * 0.05), 0.2) / 1000
            tpot = np.random.lognormal(np.log(20), 0.1) / 1000
            await asyncio.sleep(ttft + output_len * tpot)
            ttfts.append(ttft * 1000)
            tpots.append(tpot * 1000)

    t0 = time.perf_counter()
    await asyncio.gather(*[single_request() for _ in range(n_requests)])
    elapsed = time.perf_counter() - t0

    return {
        'ttft_p50': np.percentile(ttfts, 50),
        'ttft_p99': np.percentile(ttfts, 99),
        'tpot_p50': np.percentile(tpots, 50),
        'throughput_rps': n_requests / elapsed,
        'throughput_tps': n_requests * output_len / elapsed,
    }

print('vLLM benchmark simulation (100 requests, concurrency=10):')
results = asyncio.run(simulate_vllm_benchmark())
for k, v in results.items():
    print(f'  {k:<25} {v:.2f}')


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- vLLM is the standard production LLM serving framework.
- GPU memory utilization is the key vLLM knob: setting it too high causes OOM on long requests; too low wastes KV cache capacity. 0.85-0.90 is the standard starting point..
- Day 44 implementation complete.
